# 模組 5 · 交叉驗證與樣本管理
## CV 策略、分組切分、離群過濾、聚合、標記、排除

可靠的評估來自正確的驗證設計。學會選對 CV、避免資料洩漏、處理離群與重複量測。

**對應官方範例**：`examples/user/05_cross_validation/`（U01–U06）

## 🚀 環境設定（請先執行下面這格）

本課程使用 **nirs4all 官方範例資料集（sample_data）**。下面這格會：
1. 從 GitHub 下載 nirs4all（內含 0.9 版函式庫與官方光譜資料集）
2. 安裝函式庫
3. 切換到 `examples/` 目錄（裡面有 `sample_data/`）

> 💡 **為什麼從 GitHub 安裝？** PyPI（`pip install nirs4all`）目前是舊版 0.4，沒有本課程使用的 `nirs4all.run()` 簡易 API。GitHub 上的版本（0.9+）才有，且需要 **Python ≥ 3.11**（Colab 預設 3.12，沒問題）。

第一次執行約需 1–2 分鐘，請耐心等候出現「✅ 完成」。

In [ ]:
# === Colab / Jupyter 環境設定（每本 notebook 開頭執行一次）===
import os, sys, subprocess

if not os.path.isdir('nirs4all'):
    print('⬇️  下載 nirs4all 與官方資料集（約 1–2 分鐘）…')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/GBeurier/nirs4all.git'], check=True)
    print('📦 安裝 nirs4all …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    './nirs4all'], check=True)

# 切換到 examples 目錄（內含 sample_data）
if os.path.basename(os.getcwd()) != 'examples' and os.path.isdir('nirs4all/examples'):
    os.chdir('nirs4all/examples')

import nirs4all
print('✅ 完成！nirs4all 版本：', nirs4all.__version__)
print('   工作目錄：', os.getcwd())
print('   可用資料集：', [d for d in os.listdir('sample_data') if os.path.isdir(os.path.join('sample_data', d))])

---
## U01 · 交叉驗證策略  ★★☆☆☆

選對 CV 才有可靠評估：`KFold`（標準）、`ShuffleSplit`（彈性測試比例）、`RepeatedKFold`（小資料穩健）、`StratifiedKFold`（分類）。中型資料 5–10 折，折數越多越穩但越慢。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold, ShuffleSplit, RepeatedKFold
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate

for name, cv in [("KFold-5", KFold(n_splits=5, shuffle=True, random_state=42)),
                 ("ShuffleSplit", ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)),
                 ("RepeatedKFold", RepeatedKFold(n_splits=5, n_repeats=2, random_state=42))]:
    r = nirs4all.run(
        pipeline=[StandardNormalVariate(), MinMaxScaler(), {"y_processing": MinMaxScaler()},
                  cv, {"model": PLSRegression(n_components=10)}],
        dataset="sample_data/regression", name=name, verbose=0)
    print(f"{name:14s} → RMSE {r.best_rmse:.4f}")

> ✍️ **練習**：在小資料上 RepeatedKFold 的估計通常更穩定。觀察三種 CV 的 RMSE 差異。

---
## U02 · 分組切分：避免資料洩漏  ★★★☆☆

當同一樣本有多筆光譜（技術重複），若分到不同折就會**資料洩漏**、高估表現。解法：在 `DatasetConfigs` 設 `repetition="Sample_ID"`，之後任何切分器都自動尊重分組。

In [ ]:
import nirs4all
from nirs4all.dataset import DatasetConfigs
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.cross_decomposition import PLSRegression

# C04_with_metadata 含 metadata，可示範分組概念
# 一般用法：DatasetConfigs("路徑", repetition="欄位名")
pipeline = [MinMaxScaler(), {"y_processing": MinMaxScaler()},
            KFold(n_splits=3, shuffle=True, random_state=42),
            {"model": PLSRegression(n_components=10)}]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="GroupSplit", verbose=1)
print("RMSE:", round(result.best_rmse, 4))
print("\n💡 若資料有重複量測，改用：")
print('   DatasetConfigs("你的路徑", repetition="Sample_ID")')
print("   之後任何切分器都會自動按 Sample_ID 分組，避免洩漏。")

> ✍️ **練習**：對含重複量測的資料，比較「有分組」與「無分組」的測試 R²，量化洩漏幅度。

---
## U03 · 樣本排除：離群偵測與品管  ★★★☆☆

用 `exclude` 從訓練集移除離群/低品質樣本。Y 基用 `YOutlierFilter`（iqr / zscore / mad / percentile）。多過濾器可用 `mode="any"`。若只想標記不移除用 `tag`。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.filters import YOutlierFilter

# 比較：有無離群排除
for name, steps in [
    ("無排除", []),
    ("IQR 排除", [{"exclude": YOutlierFilter(method="iqr", threshold=1.5)}]),
]:
    pipeline = [MinMaxScaler(), {"y_processing": MinMaxScaler()}] + steps + [
        ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
        {"model": PLSRegression(n_components=10)}]
    r = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                     name=name, verbose=0)
    print(f"{name:10s} → RMSE {r.best_rmse:.4f}")

> ✍️ **練習**：把 threshold 從 1.5 放寬到 3.0，觀察過度排除（移除太多樣本）的風險。

---
## U04 · 重複量測聚合  ★★★☆☆

當多筆光譜代表同一樣本時，可**聚合預測**（平均）以降噪、得到每樣本一個預測。設 `repetition` 後，結果同時報告原始與聚合（標 *）指標；`top(by_repetition=True)` 用聚合排名。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression

pipeline = [
    MinMaxScaler(),
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=3)},
    {"model": PLSRegression(n_components=5)},
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="Aggregation", verbose=1)
print("\n前 3 名：")
for p in result.top(n=3, display_metrics=['rmse']):
    print(f"  {p['model_name']:18s} RMSE {p['rmse']:.4f}")
print("\n💡 若資料有 repetition 欄位，可用 result.predictions.top(3, by_repetition=True)")

> ✍️ **練習**：對有重複量測的資料，比較原始與聚合後的測試 RMSE，量化平均的降噪效益。

---
## U05 · 標記分析：標記但不移除  ★★★☆☆

`tag` 只**標記**樣本而不移除 — 樣本仍用於訓練。適合做離群影響分析、分層報告。對比：`tag` = 只標記；`exclude` = 標記並移除。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.filters import YOutlierFilter

# tag：標記離群但仍用於訓練
pipeline = [
    {"tag": YOutlierFilter(method="iqr", threshold=1.5)},
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="Tagging", verbose=1)
print("RMSE（含被標記樣本）:", round(result.best_rmse, 4))

> ✍️ **練習**：先 `tag` 再 `exclude` 同樣本，比較兩者 RMSE 以判斷是否該移除離群。

---
## U06 · 排除策略比較  ★★★★☆

系統比較排除策略：單一 vs 多過濾器、`mode="any"`（任一觸發即排除，較積極）vs `"all"`（全部觸發才排除，較保守）。排除不影響預測階段（仍預測全部樣本）。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.filters import YOutlierFilter

for name, exc in [
    ("any 模式", {"exclude": [YOutlierFilter(method="iqr", threshold=1.5),
                              YOutlierFilter(method="zscore", threshold=3.0)],
                  "mode": "any"}),
    ("all 模式", {"exclude": [YOutlierFilter(method="iqr", threshold=1.5),
                              YOutlierFilter(method="zscore", threshold=3.0)],
                  "mode": "all"}),
]:
    pipeline = [exc, MinMaxScaler(), {"y_processing": MinMaxScaler()},
                ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
                {"model": PLSRegression(n_components=10)}]
    r = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                     name=name, verbose=0)
    print(f"{name} → RMSE {r.best_rmse:.4f}")

> ✍️ **練習**：any 比 all 排除更多樣本。比較兩者最終 RMSE 與被排除樣本數的權衡。